In [1]:
!pip install yfinance pyarrow

In [2]:
from google.colab import drive
import yfinance as yf
import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm  # 使用 Colab 支援度較好的進度條
import time
import random

# ==========================================
# 1. 掛載 Google Drive (執行時會跳出 Google 授權視窗)
# ==========================================
print("🔗 正在要求掛載 Google Drive...")
drive.mount('/content/drive')

# ==========================================
# 2. 參數與資料夾設定
# ==========================================
# 預設儲存在你的雲端硬碟根目錄下的 MarketMamba_Database 資料夾
BASE_DIR = '/content/drive/MyDrive/MarketMamba_Database'
MACRO_DIR = os.path.join(BASE_DIR, "Macro_Daily")
MICRO_DIR = os.path.join(BASE_DIR, "Micro_5min")

# 自動建立資料夾 (exist_ok=True 代表如果存在就不會報錯)
for directory in [BASE_DIR, MACRO_DIR, MICRO_DIR]:
    os.makedirs(directory, exist_ok=True)

# 測試用的股票清單 (確認沒問題後，未來可以改成讀取全台股 CSV)
target_stocks = [
    "2330.TW", "2454.TW", "2317.TW", "2308.TW", "2382.TW",
    "3231.TW", "2881.TW", "2891.TW", "2603.TW", "1301.TW"
]

# ==========================================
# 3. 核心處理函數
# ==========================================
def process_data(df):
    """資料清洗與特徵計算"""
    if df is None or df.empty:
        return None

    df = df[['Close', 'Volume']].copy()
    df.fillna(method='ffill', inplace=True)

    # 計算對數報酬率與對數交易量
    df['Log_Ret'] = np.log((df['Close'] + 1e-8) / (df['Close'].shift(1) + 1e-8))
    df['Log_Vol'] = np.log(df['Volume'] + 1)

    # 清理極端值與空值
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

    return df

def download_stock_data(ticker):
    """下載單一股票並存入 Google Drive"""
    macro_path = os.path.join(MACRO_DIR, f"{ticker.replace('.TW', '')}_daily.parquet")
    micro_path = os.path.join(MICRO_DIR, f"{ticker.replace('.TW', '')}_5m.parquet")

    # 斷點續傳：如果兩個檔案都已經在雲端硬碟裡了，直接跳過
    if os.path.exists(macro_path) and os.path.exists(micro_path):
        return True

    try:
        stock = yf.Ticker(ticker)

        # 抓宏觀 (日線 1 年)
        if not os.path.exists(macro_path):
            df_daily = stock.history(period="1y", interval="1d")
            df_daily_clean = process_data(df_daily)
            if df_daily_clean is not None and len(df_daily_clean) > 100:
                df_daily_clean.to_parquet(macro_path)

        # 抓微觀 (5分線 60 天)
        if not os.path.exists(micro_path):
            df_5m = stock.history(period="60d", interval="5m")
            df_5m_clean = process_data(df_5m)
            if df_5m_clean is not None and len(df_5m_clean) > 500:
                df_5m_clean.to_parquet(micro_path)

        # 隨機暫停，避免被 Yahoo 擋 IP
        time.sleep(random.uniform(0.5, 1.5))
        return True

    except Exception as e:
        return False

# ==========================================
# 4. 執行主迴圈
# ==========================================
print(f"🚀 開始將 {len(target_stocks)} 檔股票資料寫入 Google Drive...")

success_count = 0
for ticker in tqdm(target_stocks, desc="下載進度"):
    if download_stock_data(ticker):
        success_count += 1

print(f"\n✅ 資料庫建置完成！成功抓取: {success_count}/{len(target_stocks)} 檔")
print(f"📁 請去你的 Google Drive 檢查 'MarketMamba_Database' 資料夾！")

🔗 正在要求掛載 Google Drive...
Mounted at /content/drive
🚀 開始將 10 檔股票資料寫入 Google Drive...


下載進度:   0%|          | 0/10 [00:00<?, ?it/s]

/tmp/ipykernel_236/1355375644.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
/tmp/ipykernel_236/1355375644.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
/tmp/ipykernel_236/1355375644.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
/tmp/ipykernel_236/1355375644.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
/tmp/ipykernel_236/1355375644.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj


✅ 資料庫建置完成！成功抓取: 10/10 檔
📁 請去你的 Google Drive 檢查 'MarketMamba_Database' 資料夾！


In [3]:
from google.colab import drive
import requests
import yfinance as yf
import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm
import time
import random

# ==========================================
# 1. 掛載 Google Drive
# ==========================================
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/MarketMamba_Database'
MACRO_DIR = os.path.join(BASE_DIR, "Macro_Daily")
MICRO_DIR = os.path.join(BASE_DIR, "Micro_5min")

for directory in [BASE_DIR, MACRO_DIR, MICRO_DIR]:
    os.makedirs(directory, exist_ok=True)

# ==========================================
# 2. 自動獲取全台股代號清單 (透過政府 OpenAPI)
# ==========================================
def get_all_taiwan_tickers():
    print("🔍 正在從政府開放資料平台獲取全台股代號...")
    tickers = []

    # 獲取上市公司 (上市代號後綴為 .TW)
    try:
        res_twse = requests.get("https://openapi.twse.com.tw/v1/exchangeReport/STOCK_DAY_ALL")
        data_twse = res_twse.json()
        # 過濾出長度為 4 的代號 (排除權證、牛熊證，只留普通股)
        twse_list = [f"{item['Code']}.TW" for item in data_twse if len(item['Code']) == 4]
        tickers.extend(twse_list)
    except Exception as e:
        print(f"⚠️ 上市代號獲取失敗: {e}")

    # 獲取上櫃公司 (上櫃代號後綴為 .TWO)
    try:
        res_tpex = requests.get("https://www.tpex.org.tw/openapi/v1/tpex_mainboard_quotes")
        data_tpex = res_tpex.json()
        tpex_list = [f"{item['SecuritiesCompanyCode']}.TWO" for item in data_tpex if len(item['SecuritiesCompanyCode']) == 4]
        tickers.extend(tpex_list)
    except Exception as e:
        print(f"⚠️ 上櫃代號獲取失敗: {e}")

    print(f"✅ 成功獲取 {len(tickers)} 檔普通股代號！")
    return tickers

# ==========================================
# 3. 核心處理函數 (與之前相同)
# ==========================================
def process_data(df):
    if df is None or df.empty: return None
    df = df[['Close', 'Volume']].copy()
    df.fillna(method='ffill', inplace=True)
    df['Log_Ret'] = np.log((df['Close'] + 1e-8) / (df['Close'].shift(1) + 1e-8))
    df['Log_Vol'] = np.log(df['Volume'] + 1)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    return df

def download_stock_data(ticker):
    macro_path = os.path.join(MACRO_DIR, f"{ticker.replace('.TW', '').replace('.TWO', '')}_daily.parquet")
    micro_path = os.path.join(MICRO_DIR, f"{ticker.replace('.TW', '').replace('.TWO', '')}_5m.parquet")

    # 斷點續傳
    if os.path.exists(macro_path) and os.path.exists(micro_path):
        return True

    try:
        stock = yf.Ticker(ticker)

        if not os.path.exists(macro_path):
            df_daily = stock.history(period="1y", interval="1d")
            df_daily_clean = process_data(df_daily)
            if df_daily_clean is not None and len(df_daily_clean) > 100:
                df_daily_clean.to_parquet(macro_path)

        if not os.path.exists(micro_path):
            df_5m = stock.history(period="60d", interval="5m")
            df_5m_clean = process_data(df_5m)
            if df_5m_clean is not None and len(df_5m_clean) > 500:
                df_5m_clean.to_parquet(micro_path)

        # 隨機暫停，保護 IP 不被 yfinance 封鎖
        time.sleep(random.uniform(0.3, 1.2))
        return True
    except:
        return False

# ==========================================
# 4. 執行主迴圈
# ==========================================
target_stocks = get_all_taiwan_tickers()
print(f"🚀 開始將 {len(target_stocks)} 檔股票資料寫入 Google Drive...")

success_count = 0
for ticker in tqdm(target_stocks, desc="總下載進度"):
    if download_stock_data(ticker):
        success_count += 1

print(f"\n🎉 全市場資料庫建置完成！成功抓取: {success_count}/{len(target_stocks)} 檔")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 正在從政府開放資料平台獲取全台股代號...
✅ 成功獲取 1958 檔普通股代號！
🚀 開始將 1958 檔股票資料寫入 Google Drive...


總下載進度:   0%|          | 0/1958 [00:00<?, ?it/s]

串流輸出內容已截斷至最後 5000 行。
/tmp/ipykernel_236/3426377453.py:57: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
/tmp/ipykernel_236/3426377453.py:57: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
/tmp/ipykernel_236/3426377453.py:57: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
/tmp/ipykernel_236/3426377453.py:57: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
/tmp/ipykernel_236/3426377453.py:57: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a fu


🎉 全市場資料庫建置完成！成功抓取: 1958/1958 檔
